In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
import plotly.express as px
from sklearn import preprocessing
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import scipy.cluster.hierarchy as shc
from sklearn.decomposition import PCA

In [ ]:
cs = pd.read_csv('data/customer_segmentation.csv')
cs

In [ ]:
cs = cs.drop(columns = 'Education')
cs = cs.drop(columns = 'Marital_Status')

In [ ]:
cs.head()

In [ ]:
cs.info()

In [ ]:
cs.describe().T

In [ ]:
cs['Dt_Customer'] = pd.to_datetime(cs['Dt_Customer'], format='%d-%m-%Y')


In [ ]:
corr = cs.corr()
mask = np.triu(np.ones_like(corr,dtype = bool))
plt.figure(figsize = (20,12))
sns.heatmap(corr, mask = mask)
plt.title('Correlation Heatmap', fontsize = 20)
plt.show()


In [ ]:
numerical_cols = cs.select_dtypes(exclude=['datetime64[ns]']).columns
cs_scaled = pd.DataFrame(preprocessing.StandardScaler().fit_transform(cs[numerical_cols]), columns=numerical_cols)

cs_scaled['Dt_Customer'] = cs['Dt_Customer']

cs_scaled.head()

In [ ]:
# Initialize PCA at the 3 components we want
pca = PCA(n_components = 3, random_state = 42)

# Fit PCA to the data, transform the data and then make it a dataframe we know and love
# Use pd.DataFrame instead of cs.DataFrame
#Exclude the 'Dt_Customer' column before applying PCA
# Drop rows with any missing values (NaN) before applying PCA
pca_df = pd.DataFrame(pca.fit_transform(cs_scaled.drop(columns=['Dt_Customer']).dropna()))

pca_df.shape

In [ ]:
inertias = []

for i in range(1,11):
    kmeans = KMeans(n_clusters=i, random_state=5)
    clusters = kmeans.fit_predict(pca_df)
    inertias.append(kmeans.inertia_)

plt.plot(range(1,11), inertias, marker='o')
plt.title('Elbow method')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.show()

In [ ]:
silhouette_scores = []

for i in range(2,11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    clusters = kmeans.fit_predict(pca_df)
    silhouette_scores.append(silhouette_score(pca_df, clusters))

plt.plot(range(2,11), silhouette_scores, marker='x')
plt.title('Silhouette Score')
plt.xlabel('Number of clusters')
plt.ylabel('Score')
plt.show()

In [ ]:
#cleaning up data
cs_scaled_dropped = cs_scaled.drop(columns=['Dt_Customer']).dropna()

In [ ]:
# Create the model using what we see as the optimal number of clusters
num_clusters = 3  # Replace this with your optimal number

# Create the KMeans model
kmeans = KMeans(n_clusters=num_clusters, random_state=42)

# Fit the model
kmeans.fit(cs_scaled_dropped)

# get what cluster the model thinks the points belong to and add it to our original dataframe
cs_scaled_dropped['Clusters'] = kmeans.labels_

In [ ]:
cs_scaled_dropped['Clusters'].value_counts()

In [ ]:
subset = ['ID', 'Year_Birth', 'Income',
          'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts',
          'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases',
          'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth']
# Calculate the number of rows and columns for subplots based on the subset size
num_cols = 3  # Number of columns in the subplot grid
num_rows = int(np.ceil(len(subset) / num_cols))  # Calculate the number of rows needed


fig, axs = plt.subplots(num_rows, num_cols, figsize=(30, 30))

axs = axs.flatten() # need this so you can index axs as a 1D array rather than a multidimensional array

i = 0

for column in subset:
    # Check if the column exists in cs_scaled_dropped before plotting
    if column in cs_scaled_dropped.columns:
        sns.boxplot(data=cs_scaled_dropped, y=column, x=cs_scaled_dropped["Clusters"], ax = axs[i])
        axs[i].set_title(f'Box Plot of {column}')
        axs[i].set_xlabel('Clusters')
        axs[i].set_ylabel(column)

        i = i + 1
    else:
        print(f"Skipping column '{column}' as it is not present in the dataframe.")

plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(cs_scaled_dropped, hue='Clusters', vars=subset)
plt.show()

In [ ]:
plt.figure(figsize=(20, 7))
plt.title("Dendrogram")

#lets make the dendrogram
linkage_matrix = linkage(cs_scaled_dropped, method='single')  # Change 'single' to 'ward', 'complete', or 'average' to experiment with other linkage methods
dendrogram(linkage_matrix)

plt.show()